In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 



In [ ]:
from scipy.signal import find_peaks 
from scipy import stats

def get_peaks(var):
    hist_flat = var.flatten()
    scores = stats.zscore(hist_flat.astype(np.double))
    peak_indices = []

    peak_indices_1, _ = find_peaks(-hist_flat, prominence=50, distance=20, plateau_size=1)
    peak_indices_2, _ = find_peaks(hist_flat, prominence=50, distance=20, plateau_size=1)

    peak_indices.extend([(a, -1) for a in peak_indices_1])
    peak_indices.extend([(a, 1) for a in peak_indices_2])

    filter_ind = [ind for ind, aux in peak_indices if (scores[ind] > 3 or scores[ind] < -3) and ((aux == -1 and hist_flat[ind] < 50) or (aux == 1 and hist_flat[ind] > 200))]

    return len(filter_ind)

In [ ]:
def count_split_image_all(path):
    ori_img= cv2.imread(path)
    gray = cv2.cvtColor(ori_img, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    crop_amount_x = int(width * 0.1)
    crop_amount_y = int(height * 0.1)

    gray = gray[crop_amount_y:height-crop_amount_y, crop_amount_x:width-crop_amount_x]

    var_x=np.mean(gray, axis=0).astype(np.float64)
    var_y=np.mean(gray, axis=1).astype(np.float64)

    qty_x = get_peaks(var_x)
    qty_y = get_peaks(var_y)

    return qty_x + qty_y

In [ ]:
import os
import glob

folder_path = r'/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leiomioma_animal_clean'

image_paths = glob.glob(os.path.join(folder_path, '**', '*.jpg'), recursive=True)

In [ ]:
import pandas as pd

df = pd.DataFrame(image_paths, columns=['image_path'])

In [ ]:
from tqdm import tqdm

with tqdm(total=len(df)) as pbar:
    for idx, path in enumerate(df["image_path"].values):
        count_imgs = count_split_image_all(path)
        if count_imgs == 0:
            df.loc[idx, "single"] = True
        else:
            df.loc[idx, "single"] = False
        pbar.update(1)


In [ ]:
df[df["single"] == False]

In [ ]:
df["single"].value_counts()

In [ ]:
leio_df = pd.read_csv("V2_leiomioma_animal_clean.csv")

In [ ]:
leio_df

In [ ]:
# import shutil

# single_df = df[df["single"] == False]

# with tqdm(total=len(single_df)) as pbar:
#     for _, row in single_df.iterrows():
#         # print(row["image_path"].split('/')[-1])
#         shutil.copyfile(row["image_path"], f'/media/victor/pessoal/mestrado/dataset/occ_more_cap/{row["image_path"].split('/')[-1]}')
#         pbar.update(1)

In [ ]:
import shutil

single_df = df[df["single"] == True]

with tqdm(total=len(single_df)) as pbar:
    for _, row in single_df.iterrows():
        # print(row["image_path"].split('/')[-1])
        shutil.copyfile(row["image_path"], f'/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leio_path_single/{row["image_path"].split('/')[-1]}')
        pbar.update(1)

In [ ]:
import shutil

double_df = df[df["single"] == False]

with tqdm(total=len(double_df)) as pbar:
    for _, row in double_df.iterrows():
        # print(row["image_path"].split('/')[-1])
        shutil.copyfile(row["image_path"], f'/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leio_path_double/{row["image_path"].split('/')[-1]}')
        pbar.update(1)